# ANIMA-MECHANICUS — Fregate U-ALPHA : L'Auspex Cogitateur
> *L'Ame du Mouvement, Forgee par la Machine*

**Pipeline** : `.mp4 (humain reel)` → `[Gemini (meilleur modele auto)]` → `[WHAM SMPL]` → `[Retargeting R15]` → `.npz`

**Instructions** :
1. Cellule 1 — Installer les dependances (WHAM, Torch, Detectron2, google-genai)
2. Cellule 1b — Copier SMPL_NEUTRAL.pkl depuis Google Drive *(si manquant au Pre-Flight)*
3. Cellule 2 — Configurer la cle Gemini et les parametres pipeline
4. Cellule 3 — Pre-flight check (verifier que tout est pret)
5. Cellule 4 — Uploader la video `.mp4` source
6. Cellule 5 — Lancer l'analyse Gemini et afficher le rapport de segments
7. Cellule 6 — Valider ou ajuster les segments a extraire
8. Cellule 7 — Lancer WHAM → SMPL→R15 → exporter `.npz`
9. Cellule 8 — Telecharger les `.npz` (input de la Fregate U-GAMMA)

**Runtime requis** : GPU T4 (Colab gratuit)

---

In [ ]:
# ══════ CELLULE 1 — INSTALLATION ══════
# WHAM + Torch CUDA + Detectron2 + Google Generative AI
# Duree estimee : 8-12 min sur Colab T4 (1 seule fois par session)

import os

print("[1/7] ffmpeg + libs systeme...")
os.system("apt-get install -y ffmpeg libgl1 > /dev/null 2>&1")
print("  OK")

print("[2/7] PyTorch CUDA 11.8...")
os.system("pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118")
print("  OK")

print("[3/7] mmcv + mmdet + mmpose (requis par WHAM)...")
os.system("pip install -q openmim")
os.system("mim install -q mmcv==2.0.1 mmdet==3.1.0 mmpose==1.1.0")
print("  OK")

print("[4/7] Detectron2...")
os.system("pip install -q 'git+https://github.com/facebookresearch/detectron2.git'")
print("  OK")

print("[5/7] google-generativeai...")
os.system("pip install -q google-generativeai")
print("  OK")

print("[6/7] WHAM...")
WHAM_DIR = os.path.expanduser("~/WHAM")
if not os.path.exists(WHAM_DIR):
    os.system("git clone -q https://github.com/yohanshin/WHAM.git ~/WHAM")
    os.system("pip install -q -r ~/WHAM/requirements.txt")
    print("  WHAM clone et installe")
else:
    print("  WHAM deja present")

print("[7/7] ANIMA-MECHANICUS + deps Python...")
REPO_DIR = "/content/ANIMA-MECHANICUS"
if os.path.exists(REPO_DIR):
    os.system(f"cd {REPO_DIR} && git pull -q")
else:
    os.system(f"git clone -q https://github.com/kioka8877-ux/ANIMA-MECHANICUS.git {REPO_DIR}")
os.system("pip install -q numpy scipy opencv-python-headless 'scenedetect[opencv]'")
print("  OK")

print("\n[U-ALPHA] Installation terminee.")
print("")
print("IMPORTANT — Modeles SMPL requis (si pas encore fait) :")
print("  1. S'inscrire gratuitement sur : https://smpl.is.tue.mpg.de/")
print("  2. Telecharger : SMPL_python_v.1.1.0.zip")
print("  3. Extraire SMPL_NEUTRAL.pkl")
print("  4. Uploader vers : /content/body_models/smpl/SMPL_NEUTRAL.pkl")

In [ ]:
# ══════ CELLULE 1b — SMPL DEPUIS GOOGLE DRIVE ══════
# Monte ton Drive, extrait SMPL_NEUTRAL.pkl du zip et le copie au bon endroit
# Executer seulement si le Pre-Flight signale que SMPL_NEUTRAL.pkl est manquant

import os, shutil, zipfile, time
from google.colab import drive

print("[SMPL] Montage de Google Drive...")
drive.mount("/content/drive", force_remount=False)

DST = "/content/body_models/smpl/SMPL_NEUTRAL.pkl"
os.makedirs(os.path.dirname(DST), exist_ok=True)

DRIVE_ROOT = "/content/drive/My Drive"

# ── Helper : recherche limitee pour eviter les 429 Drive API ────────────────
def _find_in_drive(root, filename_contains, extension, max_depth=3):
    """Cherche un fichier dans Drive sans traverser toute l'arborescence.

    Strategie :
      1. Racine du Drive (depth 0)
      2. Sous-dossiers directs de la racine (depth 1)
      3. Profondeur max_depth en dernier recours
    Ajoute des pauses entre les lectures de dossiers pour eviter le 429 Drive API.
    """
    found = []

    def _scan(path, depth):
        if depth > max_depth:
            return
        try:
            entries = os.listdir(path)
        except PermissionError:
            return
        time.sleep(0.05)  # pause legere : evite le 429 sur les grands Drives
        for name in entries:
            full = os.path.join(path, name)
            if os.path.isfile(full):
                n_lower = name.lower()
                if filename_contains.lower() in n_lower and n_lower.endswith(extension):
                    found.append(full)
            elif os.path.isdir(full) and depth < max_depth:
                _scan(full, depth + 1)

    _scan(root, 0)
    return found

# --- Cas 1 : SMPL_NEUTRAL.pkl deja present dans le Drive ---
print("[SMPL] Recherche de SMPL_NEUTRAL.pkl dans Drive (profondeur 3)...")
found_pkl = _find_in_drive(DRIVE_ROOT, "SMPL_NEUTRAL", ".pkl")

if found_pkl:
    shutil.copy2(found_pkl[0], DST)
    size_mb = os.path.getsize(DST) / 1024 / 1024
    print(f"[SMPL] OK — copie depuis {found_pkl[0]} ({size_mb:.1f} MB)")

else:
    # --- Cas 2 : zip SMPL present dans le Drive → extraire ---
    print("[SMPL] .pkl non trouve — recherche du zip SMPL...")
    found_zip = _find_in_drive(DRIVE_ROOT, "smpl", ".zip")

    if not found_zip:
        print("[SMPL] Aucun zip SMPL trouve dans Drive.")
        print("  Telecharger depuis : https://smpl.is.tue.mpg.de/ (inscription gratuite)")
        print("  Deposer SMPL_python_v.1.1.0.zip dans Mon Drive puis relancer cette cellule.")
        raise FileNotFoundError("SMPL zip introuvable dans Google Drive")

    zip_path = found_zip[0]
    print(f"[SMPL] Zip trouve : {zip_path}")
    print("[SMPL] Extraction de SMPL_NEUTRAL.pkl...")

    with zipfile.ZipFile(zip_path, 'r') as z:
        pkl_files = [f for f in z.namelist() if f.endswith(".pkl") and "NEUTRAL" in f.upper()]
        if not pkl_files:
            pkl_files = [f for f in z.namelist() if f.endswith(".pkl")]
        if not pkl_files:
            print("[SMPL] Contenu du zip :")
            for name in z.namelist():
                print(f"  {name}")
            raise FileNotFoundError("Aucun .pkl trouve dans le zip")

        import tempfile
        with tempfile.TemporaryDirectory() as tmp:
            z.extract(pkl_files[0], tmp)
            shutil.copy2(os.path.join(tmp, pkl_files[0]), DST)

    size_mb = os.path.getsize(DST) / 1024 / 1024
    print(f"[SMPL] OK — extrait depuis zip ({size_mb:.1f} MB)")

print(f"[SMPL] Fichier pret : {DST}")
print("[SMPL] Relancer la Cellule 3 (Pre-Flight Check)")


In [ ]:
# ══════ CELLULE 2 — CONFIGURATION ══════

import os

# @title Parametres U-ALPHA

GEMINI_API_KEY      = ""       # @param {type:"string"} — obtenir sur https://aistudio.google.com
FPS_CIBLE           = 30       # @param [30, 60, 120] {type:"raw"}
LISSAGE             = "moyen" # @param ["faible", "moyen", "brutal"] {type:"string"}
ROOT_MOTION         = True     # @param {type:"boolean"}
QUALITY_THRESHOLD   = 0.6      # @param {type:"number"}
SMPL_MODEL_PATH     = "/content/body_models/smpl/SMPL_NEUTRAL.pkl"  # @param {type:"string"}

# Cache Gemini — evite de re-consommer du quota sur la meme video
# Laisser vide "" pour desactiver, ou garder le chemin par defaut
GEMINI_CACHE_DIR    = "/content/gemini_cache"  # @param {type:"string"}

WHAM_DIR   = os.path.expanduser("~/WHAM")
REPO_DIR   = "/content/ANIMA-MECHANICUS"
OUTPUT_DIR = "/content/alpha_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
if GEMINI_CACHE_DIR:
    os.makedirs(GEMINI_CACHE_DIR, exist_ok=True)

print(f"[CONFIG] FPS={FPS_CIBLE} | Lissage={LISSAGE} | RootMotion={ROOT_MOTION} | Seuil={QUALITY_THRESHOLD}")
print(f"[CONFIG] Cache Gemini : {GEMINI_CACHE_DIR if GEMINI_CACHE_DIR else 'desactive'}")
print("[U-ALPHA] Configuration sauvegardee — passer a la Cellule 3 (pre-flight check)")


In [ ]:
# ══════ CELLULE 3 — PRE-FLIGHT CHECK ══════
# Verifie que toutes les dependances sont operationnelles avant de lancer WHAM

import os, sys, subprocess

errors  = []
warnings = []

print("[PRE-FLIGHT] Verification de l'environnement...\n")

# 1. GPU
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                             "--format=csv,noheader"], capture_output=True, text=True)
if gpu_check.returncode == 0:
    gpu_info = gpu_check.stdout.strip()
    print(f"  OK  GPU : {gpu_info}")
else:
    errors.append("GPU non detecte — activer le runtime GPU T4 dans Colab (Execution → Modifier le type d'execution)")

# 2. PyTorch + CUDA
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"  OK  PyTorch {torch.__version__} | CUDA : {'oui' if cuda_ok else 'NON'}")
    if not cuda_ok:
        warnings.append("PyTorch sans CUDA — WHAM sera tres lent (CPU only)")
except ImportError:
    errors.append("PyTorch non installe — relancer la Cellule 1")

# 3. google-generativeai
try:
    import google.generativeai
    print(f"  OK  google-generativeai {google.generativeai.__version__}")
except ImportError:
    errors.append("google-generativeai non installe — relancer la Cellule 1")

# 4. Cle Gemini
if not GEMINI_API_KEY:
    errors.append("Cle Gemini API vide — configurer GEMINI_API_KEY en Cellule 2 (https://aistudio.google.com)")
else:
    # Test leger de la cle
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        models = genai.list_models()
        print(f"  OK  Gemini API key valide")
    except Exception as e:
        errors.append(f"Gemini API key invalide ou reseau KO : {e}")

# 5. WHAM
if not os.path.exists(WHAM_DIR):
    errors.append(f"WHAM introuvable : {WHAM_DIR} — relancer Cellule 1")
else:
    demo = os.path.join(WHAM_DIR, "demo.py")
    if os.path.exists(demo):
        print(f"  OK  WHAM : {WHAM_DIR}")
    else:
        errors.append(f"WHAM incomplet (demo.py absent) — re-cloner : git clone https://github.com/yohanshin/WHAM.git ~/WHAM")

# 6. Modele SMPL
if not os.path.exists(SMPL_MODEL_PATH):
    errors.append(
        f"Modele SMPL introuvable : {SMPL_MODEL_PATH}\n"
        "  → S'inscrire sur https://smpl.is.tue.mpg.de/ et uploader SMPL_NEUTRAL.pkl"
    )
else:
    size_mb = os.path.getsize(SMPL_MODEL_PATH) / 1024 / 1024
    if size_mb < 1:
        errors.append(f"Modele SMPL trop petit ({size_mb:.1f} MB) — fichier corrompu ou incomplet")
    else:
        print(f"  OK  Modele SMPL : {SMPL_MODEL_PATH} ({size_mb:.1f} MB)")

# 7. scipy / numpy / cv2
for pkg in ["numpy", "scipy", "cv2"]:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, "__version__", "?")
        print(f"  OK  {pkg} {ver}")
    except ImportError:
        errors.append(f"{pkg} non installe — relancer Cellule 1")

# 8. ffmpeg
ffmpeg_check = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if ffmpeg_check.returncode == 0:
    version_line = ffmpeg_check.stdout.splitlines()[0]
    print(f"  OK  {version_line[:50]}")
else:
    errors.append("ffmpeg non installe — relancer Cellule 1")

# 9. Codebase ANIMA-MECHANICUS
extract_script = f"{REPO_DIR}/U-ALPHA/codebase/motus_extract.py"
if os.path.exists(extract_script):
    print(f"  OK  motus_extract.py present")
else:
    errors.append(f"motus_extract.py introuvable — relancer Cellule 1")

# ── Bilan ─────────────────────────────────────────────────────────────────
print()
if warnings:
    print("AVERTISSEMENTS :")
    for w in warnings:
        print(f"  [WARN] {w}")
    print()

if errors:
    print(f"PRE-FLIGHT ECHEC — {len(errors)} erreur(s) a corriger :")
    for i, e in enumerate(errors, 1):
        print(f"  [{i}] {e}")
    raise RuntimeError("Corriger les erreurs ci-dessus avant de continuer")
else:
    print("PRE-FLIGHT OK — Tous les systemes sont operationnels.")
    print("Passer a la Cellule 4 pour uploader la video.")

In [ ]:
# ══════ CELLULE 4 — UPLOAD VIDEO ══════
# Uploader la video .mp4 source (humain reel uniquement, max 60 secondes)

from google.colab import files as colab_files
import os, cv2

VIDEO_PATH = None

print("Option A — Upload direct depuis votre ordinateur :")
try:
    uploaded = colab_files.upload()
    for fname, data in uploaded.items():
        if fname.lower().endswith(".mp4"):
            VIDEO_PATH = f"/content/{fname}"
            with open(VIDEO_PATH, "wb") as f:
                f.write(data)
            break
except Exception:
    pass

if not VIDEO_PATH:
    VIDEO_PATH_MANUEL = ""  # @param {type:"string"}
    if VIDEO_PATH_MANUEL and os.path.exists(VIDEO_PATH_MANUEL):
        VIDEO_PATH = VIDEO_PATH_MANUEL

if not VIDEO_PATH or not os.path.exists(VIDEO_PATH):
    raise ValueError("[ERREUR] Aucune video .mp4 chargee")

cap = cv2.VideoCapture(VIDEO_PATH)
src_fps    = cap.get(cv2.CAP_PROP_FPS)
tot_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration   = tot_frames / src_fps if src_fps > 0 else 0
cap.release()
size_mb = os.path.getsize(VIDEO_PATH) / 1024 / 1024

print(f"  OK  {os.path.basename(VIDEO_PATH)}")
print(f"      {width}x{height} | {src_fps:.1f} FPS | {duration:.1f}s | {size_mb:.1f} MB")

if duration > 60:
    print(f"  [WARN] Duree {duration:.1f}s > 60s — pipeline prevu pour max 60 secondes")
if width < 480 or height < 480:
    print(f"  [WARN] Resolution faible ({width}x{height}) — qualite WHAM potentiellement reduite")

In [ ]:
# ══════ CELLULE 4b — INJECTION JSON GEMINI (MODE CHAT) ══════
# Utilise cette cellule si tu as obtenu le JSON depuis gemini.google.com (chat)
# Si tu utilises l'API automatique (Cellule 5), tu peux ignorer cette cellule.
#
# 1. Colle ton JSON dans la zone de texte
# 2. Clique "Valider et injecter"
# 3. Lance directement la Cellule 6 (Cellule 5 detectera le cache — 0 quota consomme)

import ipywidgets as widgets
from IPython.display import display
import json as _json, os, hashlib

assert VIDEO_PATH and os.path.exists(VIDEO_PATH), "[ERREUR] Charger une video en Cellule 4 d'abord"

_ta = widgets.Textarea(
    placeholder='Colle ici le JSON complet fourni par Gemini Chat...',
    layout=widgets.Layout(width='100%', height='280px')
)
_btn = widgets.Button(description='Valider et injecter', button_style='success',
                      layout=widgets.Layout(width='200px'))
_out = widgets.Output()

def _on_inject(b):
    with _out:
        _out.clear_output()
        raw = _ta.value.strip()
        if not raw:
            print("[SKIP] Zone vide — rien a injecter")
            return
        try:
            data = _json.loads(raw)
        except _json.JSONDecodeError as e:
            print(f"[ERREUR] JSON invalide : {e}")
            return

        # Calcul du nom de cache lie a la video
        cache_dir = GEMINI_CACHE_DIR or "/content/gemini_cache"
        os.makedirs(cache_dir, exist_ok=True)
        with open(VIDEO_PATH, "rb") as f:
            h = hashlib.md5(f.read(1024 * 1024)).hexdigest()[:10]
        stem = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
        cache_path = os.path.join(cache_dir, f"gemini_cache_{stem}_{h}.json")

        with open(cache_path, "w", encoding="utf-8") as f:
            _json.dump(data, f, ensure_ascii=False, indent=2)

        print(f"  JSON valide : {len(data['persons'])} personne(s) | "
              f"{data['video_duration_seconds']}s | qualite={data['qualite_globale']}")
        print()
        for p in data['persons']:
            print(f"  Personne {p['person_id']} :")
            for seg in p.get('segments_valides', []):
                modele = seg.get('modele_recommande', '?')
                extraction = seg.get('extraction_possible', '?')
                dur = seg['end_s'] - seg['start_s']
                print(f"    [{seg['start_s']}s → {seg['end_s']}s ({dur:.1f}s)] "
                      f"{seg['corps_visible']} → {modele} ({extraction})")
            for seg in p.get('segments_exclus', []):
                print(f"    [{seg['start_s']}s → {seg['end_s']}s] EXCLU : {seg['raison']}")
        print()
        print(f"  Cache sauvegarde : {cache_path}")
        print("  Lance maintenant la Cellule 5 — le cache sera detecte (0 quota consomme)")

_btn.on_click(_on_inject)
display(widgets.VBox([_ta, _btn, _out]))


In [ ]:
# ══════ CELLULE 5 — ANALYSE GEMINI (MEILLEUR MODELE AUTO) ══════

import sys, os
sys.path.insert(0, f"{REPO_DIR}/U-ALPHA/codebase")
from motus_extract import analyze_video_gemini, print_gemini_report, filter_segments

assert GEMINI_API_KEY, "[ERREUR] Configurer GEMINI_API_KEY en Cellule 2"
assert VIDEO_PATH and os.path.exists(VIDEO_PATH), "[ERREUR] Charger une video en Cellule 4"

print("[U-ALPHA] Analyse Gemini (selection automatique du meilleur modele)...")
print(f"[U-ALPHA] Depart de cascade : gemini-2.5-flash-lite (quota max free tier)")
if GEMINI_CACHE_DIR:
    print(f"[U-ALPHA] Cache actif : {GEMINI_CACHE_DIR} (re-execution gratuite sur la meme video)")

GEMINI_DATA = analyze_video_gemini(
    VIDEO_PATH,
    GEMINI_API_KEY,
    cache_dir=GEMINI_CACHE_DIR,
)
print_gemini_report(GEMINI_DATA)

ALL_SEGMENTS = filter_segments(GEMINI_DATA, QUALITY_THRESHOLD)

print(f"{'='*60}")
print(f"[FILTRE] {len(ALL_SEGMENTS)} segment(s) retenus (seuil >= {QUALITY_THRESHOLD}) :")
for s in ALL_SEGMENTS:
    dur = s['end_s'] - s['start_s']
    print(f"  P{s['person_id']} | {s['start_s']:.1f}s-{s['end_s']:.1f}s ({dur:.1f}s) | "
          f"{s['corps_visible']} | qualite={s['qualite_estimee']:.2f} | {s['type_mouvement']}")
print(f"{'='*60}")

cam = GEMINI_DATA.get("camera", {})
if cam.get("mouvement") == "agitee":
    print("[WARN] Camera agitee — root motion peut etre peu fiable")
if cam.get("zoom_detecte"):
    print("[WARN] Zoom detecte — root motion peut etre fausse")
if not ALL_SEGMENTS:
    print("[ERREUR] Aucun segment valide — verifier que la video contient un humain clairement visible")


In [ ]:
# ══════ CELLULE 6 — VALIDATION DES SEGMENTS ══════

SEGMENTS_TO_PROCESS = ALL_SEGMENTS

# === Ajustements manuels optionnels ===
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s['person_id'] == 0]
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s['start_s'] >= 5.0]
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s['corps_visible'] == 'complet']

print(f"[U-ALPHA] {len(SEGMENTS_TO_PROCESS)} segment(s) valides pour WHAM :")
total_dur = sum(s['end_s'] - s['start_s'] for s in SEGMENTS_TO_PROCESS)
for s in SEGMENTS_TO_PROCESS:
    dur = s['end_s'] - s['start_s']
    est_frames = int(dur * FPS_CIBLE)
    print(f"  P{s['person_id']} | {s['start_s']:.1f}s-{s['end_s']:.1f}s ({dur:.1f}s, ~{est_frames}f) | {s['corps_visible']}")

print(f"  Duree totale : {total_dur:.1f}s")
if not SEGMENTS_TO_PROCESS:
    raise ValueError("Aucun segment — relancer Cellule 5 ou ajuster le seuil")
else:
    print("OK — Passer a la Cellule 7 pour lancer WHAM")

In [ ]:
# ══════ CELLULE 7 — EXTRACTION WHAM → SMPL→R15 → .npz ══════
# Duree estimee : 1-3 min par segment de 30s sur Colab T4

import sys, os, tempfile
import numpy as np
import cv2

sys.path.insert(0, f"{REPO_DIR}/U-ALPHA/codebase")
from motus_extract import (
    run_wham, smpl_to_r15, fill_gaps,
    smooth_array, smooth_extremities,
    temporal_resample, normalize_quats,
    export_npz, SMOOTH_PRESETS
)

assert SEGMENTS_TO_PROCESS, "[ERREUR] Valider les segments en Cellule 6"

cap = cv2.VideoCapture(VIDEO_PATH)
src_fps    = cap.get(cv2.CAP_PROP_FPS)
tot_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration   = tot_frames / src_fps
cap.release()
print(f"[U-ALPHA] Source : {src_fps:.1f} FPS, {duration:.1f}s")

# Etape 1 : WHAM
print("\n[U-ALPHA] 1/3 — Extraction WHAM (poses SMPL)...")
with tempfile.TemporaryDirectory() as tmp_dir:
    wham_tracks = run_wham(VIDEO_PATH, SEGMENTS_TO_PROCESS, WHAM_DIR, tmp_dir)

if not wham_tracks:
    raise RuntimeError("[ERREUR] WHAM n'a produit aucun resultat")

# Etapes 2-3 : Retargeting + Lissage + Export
print("\n[U-ALPHA] 2/3 — Retargeting SMPL→R15 + Lissage...")
win, poly = SMOOTH_PRESETS[LISSAGE]
NPZ_FILES = []
n_persons = len(wham_tracks)

for pid in sorted(wham_tracks.keys()):
    track = wham_tracks[pid]
    print(f"  Personne {pid} : {len(track['poses'])} frames brutes")

    fill_gaps(track["poses"], track["transl"])
    indices  = sorted(track["poses"].keys())
    poses_aa = np.array([track["poses"][i] for i in indices])
    transl   = np.array([track["transl"][i] for i in indices])

    rots, root_pos = smpl_to_r15(poses_aa, transl)
    if not ROOT_MOTION:
        root_pos = np.zeros_like(root_pos)

    root_pos = smooth_array(root_pos, win, poly)
    rots = smooth_array(rots.reshape(len(rots), -1), win, poly).reshape(-1, 15, 4)
    rots = normalize_quats(rots)
    rots = smooth_extremities(rots, max(win * 2 + 1, 15))

    root_pos = temporal_resample(root_pos, src_fps, FPS_CIBLE)
    rots = temporal_resample(rots.reshape(len(rots), -1), src_fps, FPS_CIBLE).reshape(-1, 15, 4)
    rots = normalize_quats(rots)

    # Validation avant sauvegarde
    assert rots.shape == (len(rots), 15, 4), f"Shape rotations incorrecte : {rots.shape}"
    assert not np.any(np.isnan(rots)), "NaN detecte dans les rotations"
    assert not np.any(np.isnan(root_pos)), "NaN detecte dans root_position"

    out_path = f"{OUTPUT_DIR}/motus_core_P{pid}.npz"
    export_npz(rots, root_pos, FPS_CIBLE, src_fps, duration, pid, n_persons, out_path)
    NPZ_FILES.append(out_path)
    size_kb = os.path.getsize(out_path) / 1024
    print(f"  → {os.path.basename(out_path)} | {rots.shape[0]} frames @ {FPS_CIBLE} FPS | {size_kb:.1f} KB")

print(f"\n[U-ALPHA] {len(NPZ_FILES)} .npz exportes — Passer a la Cellule 8")

In [ ]:
# ══════ CELLULE 8 — TELECHARGEMENT DES .npz ══════

from google.colab import files as colab_files
import numpy as np, os

assert NPZ_FILES, "[ERREUR] Aucun .npz genere — relancer la Cellule 7"

print(f"[U-ALPHA] {len(NPZ_FILES)} fichier(s) .npz prets :")
for npz_path in NPZ_FILES:
    d = np.load(npz_path, allow_pickle=True)
    size_kb = os.path.getsize(npz_path) / 1024
    print(f"  {os.path.basename(npz_path)} | {d['rotations'].shape[0]} frames | {size_kb:.1f} KB")
    print(f"    rotations    : {d['rotations'].shape} {d['rotations'].dtype}")
    print(f"    root_position: {d['root_position'].shape} {d['root_position'].dtype}")
    print(f"    fps          : {int(d['fps'])} | duration : {float(d['duration']):.1f}s")

print("\nTelechargement en cours...")
for npz_path in NPZ_FILES:
    colab_files.download(npz_path)

print("\n[U-ALPHA] Mission accomplie.")
print("Prochain pas : Ouvrir ANIMA_MECHANICUS_GAMMA.ipynb")
print("  → Uploader ces .npz dans la Fregate U-GAMMA (La Forge)")